# 04 - SynthSeg reference

Runs SynthSeg 2.0 (Billot et al., 2023), as shipped with FreeSurfer 7.4.1, on all
148 scans and compares it with the models of notebook 01 on the same test folds.
SynthSeg is used as provided, with no training on this cohort.

SynthSeg does not segment the choroid plexus, so the comparison uses the other
25 structures. Its output is resampled onto the FreeSurfer label grid with
nearest-neighbour interpolation before scoring.

SynthSeg runs on the CPU here (`--cpu`, `--crop 192`), one scan per call, which
avoids running out of memory. A standard CPU runtime is enough; finished scans are
skipped, so the long step can be interrupted and re-run.

## 1. Settings, file lists and folds

In [ ]:
import os, glob, json, time, subprocess
import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.processing import resample_from_to

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive not mounted here:', e)

# same folders as notebook 01
IMAGE_DIR = '/content/drive/MyDrive/OASIS3/images'
LABEL_DIR = '/content/drive/MyDrive/OASIS3/labels'
ROOT_DIR  = '/content/drive/MyDrive/brain_seg_supervised'
SSL_DIR   = os.path.join(ROOT_DIR, 'ssl_results')               # results_fold{k}_{arm}_n{n}.json from notebook 01
SS_DIR    = os.path.join(ROOT_DIR, 'synthseg')
SEG_DIR   = os.path.join(SS_DIR, 'seg')
os.makedirs(SEG_DIR, exist_ok=True)

SEED, FOLDS = 42, 5
STD_LABEL_VALUES = [0, 2, 3, 4, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 28, 31,
                    41, 42, 43, 46, 47, 49, 50, 51, 52, 53, 60, 63]
KEEP = STD_LABEL_VALUES[1:]                                     # 27 structures (FreeSurfer values)
EVAL = [v for v in KEEP if v not in (31, 63)]                   # 25 structures (no choroid plexus)
SHORT = {2: 'L cerebral WM', 3: 'L cerebral cortex', 4: 'L lateral ventricle', 7: 'L cerebellum WM',
         8: 'L cerebellum cortex', 10: 'L thalamus', 11: 'L caudate', 12: 'L putamen', 13: 'L pallidum',
         14: '3rd ventricle', 15: '4th ventricle', 16: 'Brainstem', 17: 'L hippocampus', 28: 'L ventral DC',
         41: 'R cerebral WM', 42: 'R cerebral cortex', 43: 'R lateral ventricle', 46: 'R cerebellum WM',
         47: 'R cerebellum cortex', 49: 'R thalamus', 50: 'R caudate', 51: 'R putamen', 52: 'R pallidum',
         53: 'R hippocampus', 60: 'R ventral DC'}

image_files = sorted(glob.glob(f'{IMAGE_DIR}/**/*.nii.gz*', recursive=True))
label_files = sorted(glob.glob(f'{LABEL_DIR}/**/*.nii.gz*', recursive=True))
assert len(image_files) == len(label_files) > 0, 'images and labels not found - check IMAGE_DIR / LABEL_DIR'
n = len(image_files)
perm = np.random.RandomState(SEED).permutation(n)               # same test folds as notebook 01
fold_id = np.zeros(n, dtype=int)
for k in range(FOLDS):
    fold_id[perm[k::FOLDS]] = k
name_of = lambda p: os.path.basename(p).split('.nii')[0]
seg_of = lambda p: os.path.join(SEG_DIR, name_of(p) + '_synthseg.nii.gz')
print(f'{n} subjects | test-fold sizes: {np.bincount(fold_id).tolist()}')

## 2. Install FreeSurfer 7.4.1

Downloads the FreeSurfer 7.4.1 release (about 5 GB) to the Colab disk and unpacks
it without the example subjects. Takes 10-15 minutes. Only `mri_synthseg` is used.

In [ ]:
FS_VERSION = '7.4.1'
FREESURFER_HOME = '/content/freesurfer'
FS_URL = (f'https://surfer.nmr.mgh.harvard.edu/pub/dist/freesurfer/{FS_VERSION}/'
          f'freesurfer-linux-ubuntu22_amd64-{FS_VERSION}.tar.gz')
if not os.path.exists(os.path.join(FREESURFER_HOME, 'bin', 'mri_synthseg')):
    t0 = time.time()
    subprocess.run(f"wget -q -O - '{FS_URL}' | tar -xz -C /content "
                   "--exclude='freesurfer/subjects' --exclude='freesurfer/trctrain'",
                   shell=True, check=True)
    print(f'unpacked in {(time.time() - t0) / 60:.0f} min')
print(open(os.path.join(FREESURFER_HOME, 'build-stamp.txt')).read().strip())


def run_synthseg(img_path, out_path):
    """Segment one scan with SynthSeg 2.0 on the CPU. Returns (ok, log)."""
    cmd = (f'export FREESURFER_HOME={FREESURFER_HOME} && source $FREESURFER_HOME/SetUpFreeSurfer.sh >/dev/null && '
           f'mri_synthseg --i "{img_path}" --o "{out_path}" --cpu --threads {os.cpu_count()} --crop 192')
    r = subprocess.run(['bash', '-c', cmd], capture_output=True, text=True)
    ok = r.returncode == 0 and os.path.exists(out_path)
    return ok, (r.stdout[-2500:] + '\n' + r.stderr[-2500:])

## 3. Test on one scan

In [ ]:
t0 = time.time()
test_img = image_files[0]
if not os.path.exists(seg_of(test_img)):
    ok, log = run_synthseg(test_img, seg_of(test_img))
    print(f'ok: {ok} | {(time.time() - t0) / 60:.1f} min')
    if not ok:
        print(log)
if os.path.exists(seg_of(test_img)):
    vals = np.unique(np.asarray(nib.load(seg_of(test_img)).dataobj)).astype(int)
    print(f'{len(vals)} labels in the output | missing from our 27 structures: '
          f'{[v for v in KEEP if v not in vals]}')

## 4. All scans

One scan per call, about 1-2 min per scan on a Colab CPU.

In [ ]:
todo = [p for p in image_files if not os.path.exists(seg_of(p))]
print(f'{len(todo)} of {n} scans still to segment')
t0 = time.time()
for j, p in enumerate(todo):
    ok, log = run_synthseg(p, seg_of(p))
    if not ok:
        print('FAILED on', name_of(p)); print(log); break
    if (j + 1) % 10 == 0 or j + 1 == len(todo):
        print(f'{j + 1}/{len(todo)} | {(time.time() - t0) / (j + 1) / 60:.1f} min per scan')

## 5. Dice against the FreeSurfer labels

The SynthSeg output (1 mm, cropped) is put back onto the grid of the FreeSurfer
label map with nearest-neighbour resampling. The per-subject file stays on Drive.

In [ ]:
def dice(a, b):
    s = a.sum() + b.sum()
    return 2.0 * (a & b).sum() / s if s else np.nan


rows = []
for i in range(n):
    if not os.path.exists(seg_of(image_files[i])):
        continue
    ref_img = nib.load(label_files[i])
    ref = np.rint(np.asarray(ref_img.dataobj)).astype(np.int32)
    s = nib.load(seg_of(image_files[i]))
    if s.shape != ref.shape or not np.allclose(s.affine, ref_img.affine, atol=1e-3):
        s = resample_from_to(s, ref_img, order=0)
    seg = np.rint(np.asarray(s.dataobj)).astype(np.int32)
    row = {'subject': name_of(image_files[i]), 'fold': int(fold_id[i])}
    for v in KEEP:
        row[v] = dice(seg == v, ref == v)
    row['dice_25'] = float(np.nanmean([row[v] for v in EVAL]))
    rows.append(row)
per_subject = pd.DataFrame(rows)
per_subject.to_csv(os.path.join(SS_DIR, 'synthseg_dice_per_subject.csv'), index=False)
print(f'{len(per_subject)} subjects scored | mean Dice over 25 structures: {per_subject.dice_25.mean():.4f}')

## 6. Comparison with the trained models

Mean Dice over the 25 shared structures in each test fold, then mean and SD over the
five folds. The models' per-structure Dice comes from the result files of notebook 01.
The corrected resampled t-test (Nadeau and Bengio, 2003) compares SynthSeg with
each model across folds.

In [ ]:
from scipy import stats

RHO = 1.0 / FOLDS + 30.0 / 104.0
idx = [KEEP.index(v) for v in EVAL]


def corrected_p(d):
    d = np.asarray(d, dtype=float)
    return float(2 * stats.t.sf(abs(d.mean() / np.sqrt(RHO * d.var(ddof=1))), FOLDS - 1))


ss_fold = per_subject.groupby('fold').dice_25.mean().reindex(range(FOLDS)).values
rows = [dict(method='SynthSeg (pretrained, no training on this cohort)', n_labeled='-', dice_25_mean=ss_fold.mean(),
             dice_25_sd=ss_fold.std(ddof=1), synthseg_minus_model=np.nan, p_corr=np.nan,
             **{f'fold{k}': ss_fold[k] for k in range(FOLDS)})]
per_class = {}
for n_lab in [10, 25, 50, 104]:
    for arm, label in [('scratch', 'Random initialization'), ('ssl', 'Self-supervised pretraining')]:
        files = [os.path.join(SSL_DIR, f'results_fold{k}_{arm}_n{n_lab}.json') for k in range(FOLDS)]
        if not all(os.path.exists(f) for f in files):
            continue
        pc = np.vstack([np.asarray(json.load(open(f))['per_class_dice'], dtype=float) for f in files])
        per_class[f'{arm}_n{n_lab}'] = pc
        v = np.nanmean(pc[:, idx], axis=1)
        rows.append(dict(method=label, n_labeled=n_lab, dice_25_mean=v.mean(), dice_25_sd=v.std(ddof=1),
                         synthseg_minus_model=(ss_fold - v).mean(), p_corr=corrected_p(ss_fold - v),
                         **{f'fold{k}': v[k] for k in range(FOLDS)}))
table = pd.DataFrame(rows)
table.to_csv(os.path.join(SS_DIR, 'synthseg_vs_ours_25structures.csv'), index=False)
print(table.drop(columns=[f'fold{k}' for k in range(FOLDS)]).round(4).to_string(index=False), '\n')

# per structure: SynthSeg against the models at 25 and 104 labeled scans
ps = pd.DataFrame({'structure': [SHORT[v] for v in EVAL],
                   'SynthSeg': [per_subject[v].mean() for v in EVAL]})
for m in ['scratch_n25', 'ssl_n25', 'scratch_n104', 'ssl_n104']:
    if m in per_class:
        ps[m] = per_class[m][:, idx].mean(axis=0)
ps.to_csv(os.path.join(SS_DIR, 'synthseg_vs_ours_per_structure.csv'), index=False)
print(ps.round(4).to_string(index=False))